In [1]:
# Cell 0 | 设置项目路径和输入输出文件并构建固定的10点外部留出验证集

import hashlib
import json
import os
import shutil
import sys

from datetime import datetime
from pathlib import Path
from typing import Union

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger


RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")


def find_project_root(
    start_path: Union[str, Path] = ".",
) -> Path:
    """向上查找同时包含 data 和 notebooks 的项目根目录。"""
    current_path = Path(
        start_path
    ).resolve()

    for _ in range(20):
        if (
            (current_path / "data").exists()
            and
            (current_path / "notebooks").exists()
        ):
            return current_path

        if current_path.parent == current_path:
            break

        current_path = current_path.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认当前 notebook 位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def read_csv_robust(
    path: Path,
) -> pd.DataFrame:
    """按常见编码依次尝试读取 CSV。"""
    encodings = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]

    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                encoding=encoding,
            )

        except UnicodeDecodeError as error:
            last_error = error

    raise UnicodeDecodeError(
        last_error.encoding,
        last_error.object,
        last_error.start,
        last_error.end,
        (
            f"无法读取文件编码：{path}"
        ),
    )


def calculate_sha256(
    path: Path,
) -> str:
    """计算文件 SHA256。"""
    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(
                block
            )

    return sha256.hexdigest()


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

RAW_DIR = (
    EXPERIMENT_ROOT
    / "data"
    / "viscosity"
    / "raw"
)

VALIDATION_DIR = (
    EXPERIMENT_ROOT
    / "validation"
    / "holdout_10_points"
)

LOG_DIR = (
    EXPERIMENT_ROOT
    / "results"
    / "logs"
)

for folder in [
    RAW_DIR,
    VALIDATION_DIR,
    LOG_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# 修正后的完整24952行黏度数据
FULL_RAW_CSV = (
    RAW_DIR
    / "viscosity_full_before_holdout.csv"
)

# 修正前备份
OLD_FULL_BACKUP_CSV = (
    RAW_DIR
    / (
        "viscosity_full_before_holdout_"
        "SMILES修正前备份_20260715.csv"
    )
)

# 由本 notebook 重新生成的训练原始数据
TRAIN_RAW_CSV = (
    RAW_DIR
    / "viscosity.csv"
)

# 重建前旧训练文件的备份
OLD_TRAIN_BACKUP_CSV = (
    RAW_DIR
    / (
        "viscosity_before_SMILES_correction_"
        "rebuild_20260715.csv"
    )
)

# 原来锁定的10点
LOCKED_HOLDOUT_CSV = (
    VALIDATION_DIR
    / "holdout_10_points.csv"
)

# 原来实际删除的10条原始记录
LOCKED_REMOVED_CSV = (
    VALIDATION_DIR
    / "removed_equivalent_raw_records.csv"
)

# 本次重建生成的审计文件
SMILES_CHANGE_AUDIT_CSV = (
    VALIDATION_DIR
    / "smiles_correction_change_audit.csv"
)

FIXED_REMOVED_CSV = (
    VALIDATION_DIR
    / (
        "removed_equivalent_raw_records_"
        "fixed_rebuild.csv"
    )
)

FIXED_AUDIT_CSV = (
    VALIDATION_DIR
    / "fixed_holdout_rebuild_audit.csv"
)

FIXED_MANIFEST_JSON = (
    LOG_DIR
    / "fixed_holdout_rebuild_manifest.json"
)

required_input_files = [
    FULL_RAW_CSV,
    OLD_FULL_BACKUP_CSV,
    LOCKED_HOLDOUT_CSV,
    LOCKED_REMOVED_CSV,
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        "缺少以下输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_input_files
        )
    )

experiment_text = str(
    EXPERIMENT_ROOT.resolve()
).lower()

if (
    "small_paper_data" not in experiment_text
    or
    "viscosity_10point_holdout"
    not in experiment_text
):
    raise RuntimeError(
        "实验目录异常，已停止运行，"
        "防止覆盖大论文数据。"
    )

print("python              =", sys.executable)
print("cwd                 =", os.getcwd())
print("PROJECT_ROOT        =", PROJECT_ROOT)
print("EXPERIMENT_ROOT     =", EXPERIMENT_ROOT)

print("\n输入文件：")
print("修正后完整数据      =", FULL_RAW_CSV)
print("修正前完整数据备份  =", OLD_FULL_BACKUP_CSV)
print("锁定的10点文件      =", LOCKED_HOLDOUT_CSV)
print("锁定的删除记录      =", LOCKED_REMOVED_CSV)

print("\n输出文件：")
print("训练原始数据        =", TRAIN_RAW_CSV)
print("固定重建审计        =", FIXED_AUDIT_CSV)
print("固定重建manifest    =", FIXED_MANIFEST_JSON)

python              = D:\conda\envs\rdkit-env\python.exe
cwd                 = D:\jupyter\IL_ML_Project\notebooks\小论文_粘度预测_CO2筛选\01_10点留出验证\00_数据拆分
PROJECT_ROOT        = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT     = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout

输入文件：
修正后完整数据      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity_full_before_holdout.csv
修正前完整数据备份  = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity_full_before_holdout_SMILES修正前备份_20260715.csv
锁定的10点文件      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\holdout_10_points.csv
锁定的删除记录      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\removed_equivalent_raw_records.csv

输出文件：
训练原始数据        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity.csv
固定重建审计     

In [2]:
# Cell 1 | 读取并检查4份输入数据

df_full_new = read_csv_robust(
    FULL_RAW_CSV
)

df_full_old = read_csv_robust(
    OLD_FULL_BACKUP_CSV
)

holdout_locked = read_csv_robust(
    LOCKED_HOLDOUT_CSV
)

removed_locked = read_csv_robust(
    LOCKED_REMOVED_CSV
)

for dataframe in [
    df_full_new,
    df_full_old,
    holdout_locked,
    removed_locked,
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

raw_columns = [
    "离子液体全称",
    "cation_smiles",
    "anion_smiles",
    "temperature, k",
    "pressure, kpa",
    "Viscosity, Pa•s",
]

missing_new_columns = [
    column
    for column in raw_columns
    if column not in df_full_new.columns
]

missing_old_columns = [
    column
    for column in raw_columns
    if column not in df_full_old.columns
]

if missing_new_columns:
    raise ValueError(
        "修正后的完整数据缺少字段："
        f"{missing_new_columns}"
    )

if missing_old_columns:
    raise ValueError(
        "修正前备份缺少字段："
        f"{missing_old_columns}"
    )

required_holdout_columns = [
    "holdout_id",
    "raw_row_id",
    "离子液体全称",
    "cation_smiles",
    "anion_smiles",
    "T_K",
    "P_kPa",
    "vis_pa_s",
    "IL_key",
]

missing_holdout_columns = [
    column
    for column in required_holdout_columns
    if column not in holdout_locked.columns
]

if missing_holdout_columns:
    raise ValueError(
        "holdout_10_points.csv 缺少字段："
        f"{missing_holdout_columns}"
    )

required_removed_columns = [
    "raw_row_id",
    *raw_columns,
]

missing_removed_columns = [
    column
    for column in required_removed_columns
    if column not in removed_locked.columns
]

if missing_removed_columns:
    raise ValueError(
        "removed_equivalent_raw_records.csv "
        f"缺少字段：{missing_removed_columns}"
    )

if len(df_full_new) != 24952:
    raise ValueError(
        "修正后的完整数据行数不是24952："
        f"{len(df_full_new)}"
    )

if len(df_full_old) != 24952:
    raise ValueError(
        "修正前备份行数不是24952："
        f"{len(df_full_old)}"
    )

if len(holdout_locked) != 10:
    raise ValueError(
        "锁定的留出点数量不是10："
        f"{len(holdout_locked)}"
    )

if len(removed_locked) != 10:
    raise ValueError(
        "锁定的删除记录数量不是10："
        f"{len(removed_locked)}"
    )

holdout_locked["raw_row_id"] = pd.to_numeric(
    holdout_locked["raw_row_id"],
    errors="raise",
).astype(int)

removed_locked["raw_row_id"] = pd.to_numeric(
    removed_locked["raw_row_id"],
    errors="raise",
).astype(int)

if holdout_locked["raw_row_id"].duplicated().any():
    raise ValueError(
        "holdout_10_points.csv 中 "
        "raw_row_id 存在重复。"
    )

if removed_locked["raw_row_id"].duplicated().any():
    raise ValueError(
        "removed_equivalent_raw_records.csv 中 "
        "raw_row_id 存在重复。"
    )

print("输入数据读取完成 ✓")
print("修正后完整数据 shape =", df_full_new.shape)
print("修正前完整数据 shape =", df_full_old.shape)
print("锁定留出点数量       =", len(holdout_locked))
print("锁定删除记录数量     =", len(removed_locked))

输入数据读取完成 ✓
修正后完整数据 shape = (24952, 6)
修正前完整数据 shape = (24952, 6)
锁定留出点数量       = 10
锁定删除记录数量     = 10


In [3]:
# Cell 2 | 核对新旧完整数据只发生预期的12处SMILES修正

if df_full_new.shape != df_full_old.shape:
    raise RuntimeError(
        "新旧完整数据形状不一致。"
    )

change_records = []

for column in raw_columns:
    old_series = df_full_old[column]
    new_series = df_full_new[column]

    equal_mask = (
        old_series.eq(new_series)
        |
        (
            old_series.isna()
            & new_series.isna()
        )
    )

    changed_positions = np.where(
        ~equal_mask.to_numpy()
    )[0]

    for raw_row_id in changed_positions:
        change_records.append(
            {
                "raw_row_id": int(
                    raw_row_id
                ),
                "column": column,
                "old_value": old_series.iloc[
                    raw_row_id
                ],
                "new_value": new_series.iloc[
                    raw_row_id
                ],
            }
        )

changes_df = pd.DataFrame(
    change_records
)

if len(changes_df) != 12:
    raise RuntimeError(
        "新旧完整数据的差异单元格数量不是12："
        f"{len(changes_df)}"
    )

allowed_changed_columns = {
    "cation_smiles",
    "anion_smiles",
}

actual_changed_columns = set(
    changes_df["column"]
)

if actual_changed_columns != allowed_changed_columns:
    raise RuntimeError(
        "出现了SMILES以外的字段变化："
        f"{actual_changed_columns}"
    )

change_count_by_column = (
    changes_df["column"]
    .value_counts()
    .to_dict()
)

if change_count_by_column.get(
    "cation_smiles",
    0,
) != 5:
    raise RuntimeError(
        "阳离子SMILES修正数量不是5。"
    )

if change_count_by_column.get(
    "anion_smiles",
    0,
) != 7:
    raise RuntimeError(
        "阴离子SMILES修正数量不是7。"
    )

corrected_raw_row_ids = set(
    changes_df["raw_row_id"]
    .astype(int)
    .tolist()
)

locked_removed_raw_row_ids = set(
    removed_locked["raw_row_id"]
    .astype(int)
    .tolist()
)

holdout_correction_intersection = (
    corrected_raw_row_ids
    & locked_removed_raw_row_ids
)

if holdout_correction_intersection:
    raise RuntimeError(
        "原10点留出记录与本次SMILES修正行发生交叉："
        f"{sorted(holdout_correction_intersection)}"
    )

changes_df.to_csv(
    SMILES_CHANGE_AUDIT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("新旧完整数据差异核对通过 ✓")
print("总差异单元格数     =", len(changes_df))
print(
    "cation_smiles修正 =",
    change_count_by_column[
        "cation_smiles"
    ],
)
print(
    "anion_smiles修正  =",
    change_count_by_column[
        "anion_smiles"
    ],
)
print(
    "涉及的原始行数   =",
    changes_df["raw_row_id"].nunique(),
)
print(
    "修正行与留出行交集 =",
    len(holdout_correction_intersection),
)
print("saved:", SMILES_CHANGE_AUDIT_CSV)

display(
    changes_df
    .sort_values(
        [
            "column",
            "raw_row_id",
        ]
    )
    .reset_index(drop=True)
)

新旧完整数据差异核对通过 ✓
总差异单元格数     = 12
cation_smiles修正 = 5
anion_smiles修正  = 7
涉及的原始行数   = 12
修正行与留出行交集 = 0
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\smiles_correction_change_audit.csv


,raw_row_id,column,old_value,new_value
0,12166,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
1,12167,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
2,12168,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
3,12169,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
4,12170,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
5,12171,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
6,12172,anion_smiles,Cl[Al](Cl)(Cl)Cl->[Al-](Cl)(Cl)Cl,[Al-](Cl)(Cl)(Cl)Cl.[Al](Cl)(Cl)Cl
7,44,cation_smiles,CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,CC[S+](CC)CCOCCOCCOCCOCC[S+](CC)CC
8,45,cation_smiles,CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,CC[S+](CC)CCOCCOCCOCCOCC[S+](CC)CC
9,214,cation_smiles,C[N+]1(CCOCCOCCOCC[N+]2(C)CCCC2)CCCC1,C[N+]1(CCOCCOCCOCCOCC[N+]2(C)CCCC2)CCCC1


In [4]:
# Cell 3 | 检查修正后的全部SMILES、温压和黏度

def clean_smiles_text(
    value,
):
    """清除SMILES中的空格。"""
    if pd.isna(value):
        return np.nan

    text = (
        str(value)
        .strip()
        .replace(" ", "")
        .replace("\u3000", "")
    )

    if text == "":
        return np.nan

    return text


def canonicalize_smiles(
    smiles,
):
    """使用RDKit生成规范化SMILES。"""
    if pd.isna(smiles):
        return np.nan

    try:
        molecule = Chem.MolFromSmiles(
            str(smiles)
        )

        if molecule is None:
            return np.nan

        return Chem.MolToSmiles(
            molecule,
            canonical=True,
        )

    except Exception:
        return np.nan


def parse_pm(
    value,
):
    """解析黏度主值及不确定度。"""
    if pd.isna(value):
        return np.nan, np.nan

    text = (
        str(value)
        .strip()
        .replace("+/-", "±")
        .replace("+-", "±")
    )

    parts = re.split(
        r"\s*±\s*",
        text,
    )

    try:
        main_value = float(
            parts[0]
        )

        uncertainty = (
            float(parts[1])
            if len(parts) == 2
            else np.nan
        )

        return main_value, uncertainty

    except Exception:
        return np.nan, np.nan


import re


df_current = (
    df_full_new[
        raw_columns
    ]
    .copy()
    .reset_index()
    .rename(
        columns={
            "index": "raw_row_id",
        }
    )
)

df_current["cation_smiles_clean"] = (
    df_current["cation_smiles"]
    .map(clean_smiles_text)
)

df_current["anion_smiles_clean"] = (
    df_current["anion_smiles"]
    .map(clean_smiles_text)
)

unique_cation_smiles = (
    df_current["cation_smiles_clean"]
    .dropna()
    .drop_duplicates()
)

unique_anion_smiles = (
    df_current["anion_smiles_clean"]
    .dropna()
    .drop_duplicates()
)

cation_canonical_map = {
    smiles: canonicalize_smiles(
        smiles
    )
    for smiles in unique_cation_smiles
}

anion_canonical_map = {
    smiles: canonicalize_smiles(
        smiles
    )
    for smiles in unique_anion_smiles
}

df_current[
    "cation_smiles_canonical"
] = (
    df_current["cation_smiles_clean"]
    .map(cation_canonical_map)
)

df_current[
    "anion_smiles_canonical"
] = (
    df_current["anion_smiles_clean"]
    .map(anion_canonical_map)
)

df_current[
    [
        "vis_pa_s",
        "vis_sigma_pa_s",
    ]
] = (
    df_current[
        "Viscosity, Pa•s"
    ]
    .apply(
        lambda value: pd.Series(
            parse_pm(value)
        )
    )
)

df_current["T_K"] = pd.to_numeric(
    df_current["temperature, k"],
    errors="coerce",
)

df_current["P_kPa"] = pd.to_numeric(
    df_current["pressure, kpa"],
    errors="coerce",
)

cation_parse_failed = int(
    df_current[
        "cation_smiles_canonical"
    ].isna().sum()
)

anion_parse_failed = int(
    df_current[
        "anion_smiles_canonical"
    ].isna().sum()
)

temperature_missing = int(
    df_current["T_K"].isna().sum()
)

pressure_missing = int(
    df_current["P_kPa"].isna().sum()
)

viscosity_missing = int(
    df_current["vis_pa_s"].isna().sum()
)

if cation_parse_failed != 0:
    raise RuntimeError(
        "仍有阳离子SMILES无法被RDKit解析："
        f"{cation_parse_failed}"
    )

if anion_parse_failed != 0:
    raise RuntimeError(
        "仍有阴离子SMILES无法被RDKit解析："
        f"{anion_parse_failed}"
    )

if temperature_missing != 0:
    raise RuntimeError(
        "温度存在缺失："
        f"{temperature_missing}"
    )

if pressure_missing != 0:
    raise RuntimeError(
        "压力存在缺失："
        f"{pressure_missing}"
    )

if viscosity_missing != 0:
    raise RuntimeError(
        "黏度主值存在解析失败："
        f"{viscosity_missing}"
    )

# 进一步核查5条四氧连接链修正
cation_change_rows = (
    changes_df[
        changes_df["column"].eq(
            "cation_smiles"
        )
    ]
    .copy()
)

for _, change_row in (
    cation_change_rows.iterrows()
):
    corrected_smiles = str(
        change_row["new_value"]
    )

    molecule = Chem.MolFromSmiles(
        corrected_smiles
    )

    oxygen_count = sum(
        1
        for atom in molecule.GetAtoms()
        if atom.GetSymbol() == "O"
    )

    formal_charge = Chem.GetFormalCharge(
        molecule
    )

    if oxygen_count != 4:
        raise RuntimeError(
            "四氧连接链修正后的氧原子数不是4：\n"
            f"raw_row_id="
            f"{change_row['raw_row_id']}"
        )

    if formal_charge != 2:
        raise RuntimeError(
            "四氧连接链修正后的形式电荷不是+2：\n"
            f"raw_row_id="
            f"{change_row['raw_row_id']}"
        )

print("修正后完整数据结构和数值检查通过 ✓")
print("完整记录数            =", len(df_current))
print("唯一阳离子SMILES数    =", len(unique_cation_smiles))
print("唯一阴离子SMILES数    =", len(unique_anion_smiles))
print("阳离子解析失败数      =", cation_parse_failed)
print("阴离子解析失败数      =", anion_parse_failed)
print("温度缺失数            =", temperature_missing)
print("压力缺失数            =", pressure_missing)
print("黏度解析失败数        =", viscosity_missing)
print("5条四氧连接链核验     = 全部通过")

修正后完整数据结构和数值检查通过 ✓
完整记录数            = 24952
唯一阳离子SMILES数    = 584
唯一阴离子SMILES数    = 202
阳离子解析失败数      = 0
阴离子解析失败数      = 0
温度缺失数            = 0
压力缺失数            = 0
黏度解析失败数        = 0
5条四氧连接链核验     = 全部通过


In [5]:
# Cell 4 | 核对原来的10个留出点及删除记录

holdout_raw_ids = set(
    holdout_locked["raw_row_id"]
    .astype(int)
    .tolist()
)

removed_raw_ids = set(
    removed_locked["raw_row_id"]
    .astype(int)
    .tolist()
)

if holdout_raw_ids != removed_raw_ids:
    raise RuntimeError(
        "holdout_10_points.csv 与 "
        "removed_equivalent_raw_records.csv "
        "中的 raw_row_id 集合不一致。"
    )

if len(holdout_raw_ids) != 10:
    raise RuntimeError(
        "锁定的唯一 raw_row_id 数量不是10。"
    )

invalid_raw_ids = [
    raw_row_id
    for raw_row_id in sorted(
        removed_raw_ids
    )
    if (
        raw_row_id < 0
        or raw_row_id >= len(df_current)
    )
]

if invalid_raw_ids:
    raise RuntimeError(
        "存在超出当前完整数据范围的 raw_row_id："
        f"{invalid_raw_ids}"
    )

removed_lookup = (
    removed_locked
    .set_index("raw_row_id")
)

holdout_lookup = (
    holdout_locked
    .set_index("raw_row_id")
)

audit_rows = []

for raw_row_id in sorted(
    removed_raw_ids
):
    current_row = (
        df_current.loc[
            df_current[
                "raw_row_id"
            ].eq(raw_row_id)
        ]
        .iloc[0]
    )

    locked_removed_row = (
        removed_lookup.loc[
            raw_row_id
        ]
    )

    locked_holdout_row = (
        holdout_lookup.loc[
            raw_row_id
        ]
    )

    name_match = (
        str(
            current_row[
                "离子液体全称"
            ]
        )
        ==
        str(
            locked_removed_row[
                "离子液体全称"
            ]
        )
    )

    cation_match = (
        str(
            current_row[
                "cation_smiles"
            ]
        )
        ==
        str(
            locked_removed_row[
                "cation_smiles"
            ]
        )
    )

    anion_match = (
        str(
            current_row[
                "anion_smiles"
            ]
        )
        ==
        str(
            locked_removed_row[
                "anion_smiles"
            ]
        )
    )

    temperature_match = np.isclose(
        float(
            current_row["T_K"]
        ),
        float(
            locked_removed_row[
                "temperature, k"
            ]
        ),
        rtol=0.0,
        atol=1e-12,
    )

    pressure_match = np.isclose(
        float(
            current_row["P_kPa"]
        ),
        float(
            locked_removed_row[
                "pressure, kpa"
            ]
        ),
        rtol=0.0,
        atol=1e-12,
    )

    viscosity_text_match = (
        str(
            current_row[
                "Viscosity, Pa•s"
            ]
        )
        ==
        str(
            locked_removed_row[
                "Viscosity, Pa•s"
            ]
        )
    )

    holdout_temperature_match = np.isclose(
        float(
            current_row["T_K"]
        ),
        float(
            locked_holdout_row[
                "T_K"
            ]
        ),
        rtol=0.0,
        atol=1e-12,
    )

    holdout_pressure_match = np.isclose(
        float(
            current_row["P_kPa"]
        ),
        float(
            locked_holdout_row[
                "P_kPa"
            ]
        ),
        rtol=0.0,
        atol=1e-12,
    )

    holdout_viscosity_match = np.isclose(
        float(
            current_row["vis_pa_s"]
        ),
        float(
            locked_holdout_row[
                "vis_pa_s"
            ]
        ),
        rtol=0.0,
        atol=1e-12,
    )

    all_removed_fields_match = all(
        [
            name_match,
            cation_match,
            anion_match,
            temperature_match,
            pressure_match,
            viscosity_text_match,
        ]
    )

    all_holdout_fields_match = all(
        [
            holdout_temperature_match,
            holdout_pressure_match,
            holdout_viscosity_match,
        ]
    )

    audit_rows.append(
        {
            "holdout_id": (
                locked_holdout_row[
                    "holdout_id"
                ]
            ),
            "raw_row_id": raw_row_id,
            "离子液体全称": (
                current_row[
                    "离子液体全称"
                ]
            ),
            "IL_key": (
                locked_holdout_row[
                    "IL_key"
                ]
            ),
            "T_K": float(
                current_row["T_K"]
            ),
            "P_kPa": float(
                current_row["P_kPa"]
            ),
            "vis_pa_s": float(
                current_row["vis_pa_s"]
            ),
            "name_match": name_match,
            "cation_match": cation_match,
            "anion_match": anion_match,
            "temperature_match": (
                temperature_match
            ),
            "pressure_match": (
                pressure_match
            ),
            "viscosity_text_match": (
                viscosity_text_match
            ),
            "holdout_T_match": (
                holdout_temperature_match
            ),
            "holdout_P_match": (
                holdout_pressure_match
            ),
            "holdout_viscosity_match": (
                holdout_viscosity_match
            ),
            "matches_locked_removed_record": (
                all_removed_fields_match
            ),
            "matches_locked_holdout_point": (
                all_holdout_fields_match
            ),
            "is_smiles_corrected_row": (
                raw_row_id
                in corrected_raw_row_ids
            ),
            "fixed_remove": True,
        }
    )

fixed_audit_df = (
    pd.DataFrame(
        audit_rows
    )
    .sort_values(
        "holdout_id"
    )
    .reset_index(drop=True)
)

if not fixed_audit_df[
    "matches_locked_removed_record"
].all():
    raise RuntimeError(
        "至少有一个当前原始记录与原删除记录不一致。"
    )

if not fixed_audit_df[
    "matches_locked_holdout_point"
].all():
    raise RuntimeError(
        "至少有一个当前原始记录与锁定留出点不一致。"
    )

if fixed_audit_df[
    "is_smiles_corrected_row"
].any():
    raise RuntimeError(
        "原留出点中出现本次SMILES修正记录。"
    )

holdout_sha256_before = calculate_sha256(
    LOCKED_HOLDOUT_CSV
)

removed_sha256_before = calculate_sha256(
    LOCKED_REMOVED_CSV
)

print("原10点留出集核对通过 ✓")
print("锁定留出点数          =", len(holdout_locked))
print("锁定删除记录数        =", len(removed_locked))
print("raw_row_id集合一致    =", True)
print(
    "10点与当前完整数据一致 =",
    fixed_audit_df[
        "matches_locked_removed_record"
    ].all(),
)
print(
    "10点涉及SMILES修正行  =",
    int(
        fixed_audit_df[
            "is_smiles_corrected_row"
        ].sum()
    ),
)

display(
    fixed_audit_df
)

原10点留出集核对通过 ✓
锁定留出点数          = 10
锁定删除记录数        = 10
raw_row_id集合一致    = True
10点与当前完整数据一致 = True
10点涉及SMILES修正行  = 0


,holdout_id,raw_row_id,离子液体全称,IL_key,T_K,P_kPa,vis_pa_s,name_match,cation_match,anion_match,temperature_match,pressure_match,viscosity_text_match,holdout_T_match,holdout_P_match,holdout_viscosity_match,matches_locked_removed_record,matches_locked_holdout_point,is_smiles_corrected_row,fixed_remove
0,H01,8190,1-butyl-3-methylpyridinium tricyanomethanide,CCCC[n+]1cccc(C)c1||N#CC(=C=[N-])C#N,313.15,101.325,0.02327,True,True,True,True,True,True,True,True,True,True,True,False,True
1,H02,1034,1-(2-ethoxyethyl)pyridinium bis((trifluorometh...,CCOCC[n+]1ccccc1||O=S(=O)([N-]S(=O)(=O)C(F)(F)...,313.15,101.325,0.02840,True,True,True,True,True,True,True,True,True,True,True,False,True
2,H03,862,1-(2-chloroethyl)-3-methylimidazolium dicyanamide,C[n+]1ccn(CCCl)c1||N#C[N-]C#N,328.15,101.325,0.03110,True,True,True,True,True,True,True,True,True,True,True,False,True
3,H04,15544,"1-hexyl-4-(4-sulfobutyl)-1H-1,2,4-triazolium t...",CCCCCCn1cn[n+](CCCCS(=O)(=O)O)c1||O=S(=O)([O-]...,318.15,100.000,0.05110,True,True,True,True,True,True,True,True,True,True,True,False,True
4,H05,22966,tributyldecylphosphonium bis(trifluoromethanes...,CCCCCCCCCC[P+](CCCC)(CCCC)CCCC||O=S(=O)([N-]S(...,328.10,101.325,0.07000,True,True,True,True,True,True,True,True,True,True,True,False,True
5,H06,21059,"N-isopropyl-N,N-dimethylhexan-1-aminium bis((t...",CCCCCC[N+](C)(C)C(C)C||O=S(=O)([N-]S(=O)(=O)C(...,308.10,101.325,0.08900,True,True,True,True,True,True,True,True,True,True,True,False,True
6,H07,16718,1-methyl-3-octylimidazolium tricyanomethanide,CCCCCCCCn1cc[n+](C)c1||N#CC(=C=[N-])C#N,283.20,100.000,0.14000,True,True,True,True,True,True,True,True,True,True,True,False,True
7,H08,23326,triethyl(pentyl)phosphonium bis((perfluoroethy...,CCCCC[P+](CC)(CC)CC||O=S(=O)([N-]S(=O)(=O)C(F)...,303.15,101.325,0.15070,True,True,True,True,True,True,True,True,True,True,True,False,True
8,H09,3855,1-butyl-3-methylimidazolium L-serinate,CCCCn1cc[n+](C)c1||NC(CO)C(=O)[O-],318.15,101.325,1.05000,True,True,True,True,True,True,True,True,True,True,True,False,True
9,H10,21218,N-methyl-N-octylmorpholinium N-acetyl-L-alaninate,CCCCCCCC[N+]1(C)CCOCC1||CC(=O)N[C@@H](C)C(=O)[O-],308.15,101.325,9.70000,True,True,True,True,True,True,True,True,True,True,True,False,True


In [6]:
# Cell 5 | 固定删除原来的10条记录并重新生成训练数据

# 在首次重建前备份旧的训练原始数据
if (
    TRAIN_RAW_CSV.exists()
    and not OLD_TRAIN_BACKUP_CSV.exists()
):
    shutil.copy2(
        TRAIN_RAW_CSV,
        OLD_TRAIN_BACKUP_CSV,
    )

    print(
        "已备份旧训练原始数据：",
        OLD_TRAIN_BACKUP_CSV,
    )

elif OLD_TRAIN_BACKUP_CSV.exists():
    print(
        "旧训练数据备份已存在，不重复覆盖：",
        OLD_TRAIN_BACKUP_CSV,
    )

remove_mask = (
    df_current["raw_row_id"]
    .isin(
        sorted(
            removed_raw_ids
        )
    )
)

removed_current = (
    df_current.loc[
        remove_mask,
        [
            "raw_row_id",
            *raw_columns,
        ],
    ]
    .sort_values(
        "raw_row_id"
    )
    .reset_index(drop=True)
)

train_raw = (
    df_current.loc[
        ~remove_mask,
        raw_columns,
    ]
    .copy()
    .reset_index(drop=True)
)

if len(removed_current) != 10:
    raise RuntimeError(
        "本次实际删除记录数不是10："
        f"{len(removed_current)}"
    )

if len(train_raw) != 24942:
    raise RuntimeError(
        "删除10条后训练原始数据不是24942行："
        f"{len(train_raw)}"
    )

train_raw.to_csv(
    TRAIN_RAW_CSV,
    index=False,
    encoding="utf-8-sig",
)

removed_current.to_csv(
    FIXED_REMOVED_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("固定删除原10点完成 ✓")
print("留出前完整记录数 =", len(df_current))
print("实际删除记录数   =", len(removed_current))
print("删除后训练记录数 =", len(train_raw))
print("saved:", TRAIN_RAW_CSV)
print("saved:", FIXED_REMOVED_CSV)

display(
    removed_current
)

已备份旧训练原始数据： D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity_before_SMILES_correction_rebuild_20260715.csv
固定删除原10点完成 ✓
留出前完整记录数 = 24952
实际删除记录数   = 10
删除后训练记录数 = 24942
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\removed_equivalent_raw_records_fixed_rebuild.csv


,raw_row_id,离子液体全称,cation_smiles,anion_smiles,"temperature, k","pressure, kpa","Viscosity, Pa•s"
0,862,1-(2-chloroethyl)-3-methylimidazolium dicyanamide,C[n+]1ccn(CCCl)c1,N#C[N-]C#N,328.15,101.325,0.0311 ± 0.0032
1,1034,1-(2-ethoxyethyl)pyridinium bis((trifluorometh...,CCOCC[n+]1ccccc1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,313.15,101.325,0.0284 ± 0.0007
2,3855,1-butyl-3-methylimidazolium L-serinate,CCCCn1cc[n+](C)c1,NC(CO)C(=O)[O-],318.15,101.325,1.05 ± 0.11
3,8190,1-butyl-3-methylpyridinium tricyanomethanide,CCCC[n+]1cccc(C)c1,N#CC(=C=[N-])C#N,313.15,101.325,0.02327 ± 0.00098
4,15544,"1-hexyl-4-(4-sulfobutyl)-1H-1,2,4-triazolium t...",CCCCCCn1cn[n+](CCCCS(=O)(=O)O)c1,O=S(=O)([O-])C(F)(F)F,318.15,100.000,0.0511 ± 0
5,16718,1-methyl-3-octylimidazolium tricyanomethanide,CCCCCCCCn1cc[n+](C)c1,N#CC(=C=[N-])C#N,283.20,100.000,0.14 ± 0.02
6,21059,"N-isopropyl-N,N-dimethylhexan-1-aminium bis((t...",CCCCCC[N+](C)(C)C(C)C,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,308.10,101.325,0.089 ± 0.018
7,21218,N-methyl-N-octylmorpholinium N-acetyl-L-alaninate,CCCCCCCC[N+]1(C)CCOCC1,CC(=O)N[C@@H](C)C(=O)[O-],308.15,101.325,9.7 ± 1.7
8,22966,tributyldecylphosphonium bis(trifluoromethanes...,CCCCCCCCCC[P+](CCCC)(CCCC)CCCC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,328.10,101.325,0.07 ± 0.0032
9,23326,triethyl(pentyl)phosphonium bis((perfluoroethy...,CCCCC[P+](CC)(CC)CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)C(F)(F)F)C(F)(F)C(...,303.15,101.325,0.1507 ± 0.0037


In [7]:
# Cell 6 | 保存固定重建审计和manifest

# 重新读取刚保存的训练文件并逐项核对
saved_train_raw = read_csv_robust(
    TRAIN_RAW_CSV
)

saved_train_raw.columns = (
    saved_train_raw.columns
    .astype(str)
    .str.strip()
)

if saved_train_raw.shape != (
    24942,
    6,
):
    raise RuntimeError(
        "保存后的 viscosity.csv 形状错误："
        f"{saved_train_raw.shape}"
    )

expected_train_raw = (
    df_full_new.loc[
        ~df_full_new.index.isin(
            sorted(
                removed_raw_ids
            )
        ),
        raw_columns,
    ]
    .reset_index(drop=True)
)

for column in raw_columns:
    saved_series = (
        saved_train_raw[column]
    )

    expected_series = (
        expected_train_raw[column]
    )

    if column in [
        "temperature, k",
        "pressure, kpa",
    ]:
        column_match = np.allclose(
            pd.to_numeric(
                saved_series,
                errors="raise",
            ),
            pd.to_numeric(
                expected_series,
                errors="raise",
            ),
            rtol=0.0,
            atol=1e-12,
        )

    else:
        column_match = bool(
            (
                saved_series.astype(str)
                ==
                expected_series.astype(str)
            ).all()
        )

    if not column_match:
        raise RuntimeError(
            "保存后的训练数据字段不一致："
            f"{column}"
        )

fixed_audit_df.to_csv(
    FIXED_AUDIT_CSV,
    index=False,
    encoding="utf-8-sig",
)

holdout_sha256_after = calculate_sha256(
    LOCKED_HOLDOUT_CSV
)

removed_sha256_after = calculate_sha256(
    LOCKED_REMOVED_CSV
)

if (
    holdout_sha256_after
    != holdout_sha256_before
):
    raise RuntimeError(
        "锁定的 holdout_10_points.csv "
        "在运行过程中被修改。"
    )

if (
    removed_sha256_after
    != removed_sha256_before
):
    raise RuntimeError(
        "原 removed_equivalent_raw_records.csv "
        "在运行过程中被修改。"
    )

manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "experiment_root": str(
        EXPERIMENT_ROOT
    ),

    "task": (
        "fixed_original_10point_holdout_"
        "rebuild_after_smiles_correction"
    ),

    "holdout_mode": (
        "locked_original_raw_row_ids"
    ),

    "holdout_selection_performed": False,

    "random_sampling_performed": False,

    "inputs": {
        "corrected_full_raw": str(
            FULL_RAW_CSV
        ),
        "old_full_backup": str(
            OLD_FULL_BACKUP_CSV
        ),
        "locked_holdout": str(
            LOCKED_HOLDOUT_CSV
        ),
        "locked_removed_records": str(
            LOCKED_REMOVED_CSV
        ),
    },

    "input_hashes": {
        "corrected_full_raw_sha256": (
            calculate_sha256(
                FULL_RAW_CSV
            )
        ),
        "old_full_backup_sha256": (
            calculate_sha256(
                OLD_FULL_BACKUP_CSV
            )
        ),
        "locked_holdout_sha256": (
            holdout_sha256_after
        ),
        "locked_removed_records_sha256": (
            removed_sha256_after
        ),
    },

    "smiles_correction_check": {
        "changed_cells": int(
            len(changes_df)
        ),
        "changed_raw_rows": int(
            changes_df[
                "raw_row_id"
            ].nunique()
        ),
        "cation_smiles_changes": int(
            change_count_by_column[
                "cation_smiles"
            ]
        ),
        "anion_smiles_changes": int(
            change_count_by_column[
                "anion_smiles"
            ]
        ),
        "non_smiles_field_changes": 0,
        "corrected_rows_in_holdout": int(
            len(
                holdout_correction_intersection
            )
        ),
        "change_audit_file": str(
            SMILES_CHANGE_AUDIT_CSV
        ),
    },

    "structure_check": {
        "cation_rdkit_parse_failed": int(
            cation_parse_failed
        ),
        "anion_rdkit_parse_failed": int(
            anion_parse_failed
        ),
        "tetraoxa_corrected_rows_checked": 5,
    },

    "counts": {
        "full_raw_rows": int(
            len(df_full_new)
        ),
        "locked_holdout_points": int(
            len(holdout_locked)
        ),
        "locked_removed_raw_records": int(
            len(removed_current)
        ),
        "training_raw_rows": int(
            len(train_raw)
        ),
    },

    "holdout_audit": {
        "all_raw_ids_preserved": True,
        "all_locked_records_matched": bool(
            fixed_audit_df[
                "matches_locked_removed_record"
            ].all()
        ),
        "all_locked_points_matched": bool(
            fixed_audit_df[
                "matches_locked_holdout_point"
            ].all()
        ),
        "audit_file": str(
            FIXED_AUDIT_CSV
        ),
    },

    "outputs": {
        "training_raw_csv": str(
            TRAIN_RAW_CSV
        ),
        "training_raw_sha256": (
            calculate_sha256(
                TRAIN_RAW_CSV
            )
        ),
        "fixed_removed_records": str(
            FIXED_REMOVED_CSV
        ),
        "fixed_rebuild_audit": str(
            FIXED_AUDIT_CSV
        ),
        "old_training_backup": str(
            OLD_TRAIN_BACKUP_CSV
        ),
    },
}

with open(
    FIXED_MANIFEST_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("固定重建审计和manifest保存完成 ✓")
print("saved:", FIXED_AUDIT_CSV)
print("saved:", FIXED_MANIFEST_JSON)

固定重建审计和manifest保存完成 ✓
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\fixed_holdout_rebuild_audit.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\results\logs\fixed_holdout_rebuild_manifest.json


In [8]:
# Cell 7 | 最终检查并输出完整结果

expected_output_files = [
    TRAIN_RAW_CSV,
    SMILES_CHANGE_AUDIT_CSV,
    FIXED_REMOVED_CSV,
    FIXED_AUDIT_CSV,
    FIXED_MANIFEST_JSON,
]

missing_output_files = [
    path
    for path in expected_output_files
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "以下输出文件缺失：\n"
        + "\n".join(
            str(path)
            for path in missing_output_files
        )
    )

train_check = read_csv_robust(
    TRAIN_RAW_CSV
)

change_audit_check = read_csv_robust(
    SMILES_CHANGE_AUDIT_CSV
)

removed_check = read_csv_robust(
    FIXED_REMOVED_CSV
)

fixed_audit_check = read_csv_robust(
    FIXED_AUDIT_CSV
)

with open(
    FIXED_MANIFEST_JSON,
    "r",
    encoding="utf-8",
) as file:
    manifest_check = json.load(
        file
    )

if train_check.shape != (
    24942,
    6,
):
    raise RuntimeError(
        "最终训练文件形状错误："
        f"{train_check.shape}"
    )

if len(change_audit_check) != 12:
    raise RuntimeError(
        "SMILES修正审计记录数不是12。"
    )

if len(removed_check) != 10:
    raise RuntimeError(
        "固定删除记录数不是10。"
    )

if len(fixed_audit_check) != 10:
    raise RuntimeError(
        "固定留出审计记录数不是10。"
    )

if not fixed_audit_check[
    "matches_locked_removed_record"
].astype(bool).all():
    raise RuntimeError(
        "存在未匹配原删除记录的留出点。"
    )

if not fixed_audit_check[
    "matches_locked_holdout_point"
].astype(bool).all():
    raise RuntimeError(
        "存在未匹配锁定留出点的记录。"
    )

if fixed_audit_check[
    "is_smiles_corrected_row"
].astype(bool).any():
    raise RuntimeError(
        "本次修正行错误地进入原10点留出集。"
    )

if (
    manifest_check.get(
        "holdout_selection_performed"
    )
    is not False
):
    raise RuntimeError(
        "manifest 错误地标记为重新选择留出点。"
    )

if (
    manifest_check.get(
        "random_sampling_performed"
    )
    is not False
):
    raise RuntimeError(
        "manifest 错误地标记为执行了随机抽样。"
    )

if (
    calculate_sha256(
        LOCKED_HOLDOUT_CSV
    )
    != holdout_sha256_before
):
    raise RuntimeError(
        "原 holdout_10_points.csv 被修改。"
    )

if (
    calculate_sha256(
        LOCKED_REMOVED_CSV
    )
    != removed_sha256_before
):
    raise RuntimeError(
        "原 removed_equivalent_raw_records.csv 被修改。"
    )

print("=" * 84)
print("固定原10点留出集重建检查全部通过 ✓")
print("=" * 84)

print("\n完整数据修正：")
print("完整原始记录数        =", len(df_full_new))
print("SMILES差异单元格数    =", len(change_audit_check))
print("cation_smiles修正数   =", 5)
print("anion_smiles修正数    =", 7)
print("非SMILES字段变化数    =", 0)
print("修正行与留出行交集    =", 0)

print("\n结构检查：")
print("阳离子解析失败数      =", cation_parse_failed)
print("阴离子解析失败数      =", anion_parse_failed)
print("四氧连接链核验数      =", 5)

print("\n固定留出集：")
print("重新选择留出点        =", False)
print("执行随机抽样          =", False)
print("锁定留出点数          =", len(holdout_locked))
print("实际固定删除记录数    =", len(removed_check))
print("删除后训练记录数      =", len(train_check))
print(
    "全部原留出点匹配通过  =",
    bool(
        fixed_audit_check[
            "matches_locked_holdout_point"
        ].astype(bool).all()
    ),
)
print(
    "原留出文件保持未修改  =",
    True,
)

print("\n输出文件完整路径：")
for path in expected_output_files:
    print(path)

if OLD_TRAIN_BACKUP_CSV.exists():
    print(OLD_TRAIN_BACKUP_CSV)

固定原10点留出集重建检查全部通过 ✓

完整数据修正：
完整原始记录数        = 24952
SMILES差异单元格数    = 12
cation_smiles修正数   = 5
anion_smiles修正数    = 7
非SMILES字段变化数    = 0
修正行与留出行交集    = 0

结构检查：
阳离子解析失败数      = 0
阴离子解析失败数      = 0
四氧连接链核验数      = 5

固定留出集：
重新选择留出点        = False
执行随机抽样          = False
锁定留出点数          = 10
实际固定删除记录数    = 10
删除后训练记录数      = 24942
全部原留出点匹配通过  = True
原留出文件保持未修改  = True

输出文件完整路径：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\smiles_correction_change_audit.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\removed_equivalent_raw_records_fixed_rebuild.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\validation\holdout_10_points\fixed_holdout_rebuild_audit.csv
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\results\logs\fixed_holdout_rebuild_manifes